In [1]:
# gemma-2-2b_batch_top_k_width-2pow14_date-0107/resid_post_layer_12/trainer_9 -> BatchTopK, k=160
# gemma-2-2b_batch_top_k_width-2pow14_date-0107/resid_post_layer_12/trainer_8 -> BatchTopK, k=80
# context is 1024

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# This works on an A40 (48GB VRAM); may need to adjust for other hardware
EVAL_BATCH_SIZE = 8
TOKENIZER_BATCH_SIZE = 256
DEVICE = "cuda:0"

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [3]:
from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

eval_layer = 12
saes = {}
for sae_spec in ("next_layer_finetuned_interaction",):
    checkpoint_dir = f"/workspace/sae_checkpoints/gemma_2_2b/{sae_spec}"
    checkpoint = find_latest_checkpoint(checkpoint_dir, eval_layer)
    saes[sae_spec] = load_checkpoint(checkpoint).sae

In [25]:
from transformers_sae.sae_bench import run_sae_bench_evals

eval_results = run_sae_bench_evals(
    model,
    tokenizer,
    saes["next_layer_finetuned_interaction"],
    eval_layer,
    batch_size=EVAL_BATCH_SIZE,
    num_tokens=int(1e3),
    dataset=validation_dataset,
)
print(eval_results[0])

Reconstruction Batches:   0%|                                                               | 0/1 [00:00<?, ?it/s]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])


Sparsity and Variance Batches:   0%|                                                        | 0/1 [00:00<?, ?it/s]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])


Sparsity and Variance Batches: 100%|████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.34s/it]


{'model_behavior_preservation': {'kl_div_score': 0.9467147435897436, 'kl_div_with_ablation': 9.75, 'kl_div_with_sae': 0.51953125}, 'model_performance_preservation': {'ce_loss_score': 1.00418410041841, 'ce_loss_with_ablation': 12.4375, 'ce_loss_with_sae': 4.9375, 'ce_loss_without_sae': 4.96875}, 'reconstruction_quality': {'explained_variance': 0.69140625, 'explained_variance_legacy': 0.376953125, 'mse': 0.6484375, 'cossim': 0.83203125}, 'shrinkage': {'l2_norm_in': 129.0, 'l2_norm_out': 111.5, 'l2_ratio': 0.86328125, 'relative_reconstruction_bias': 1.0390625}, 'sparsity': {'l0': 192.55726623535156, 'l1': 664.0}, 'token_stats': {'total_tokens_eval_reconstruction': 8192, 'total_tokens_eval_sparsity_variance': 8192}}


In [8]:
model.config._attn_implementation

'sdpa'

In [35]:
from transformers_sae.validation import run_validations
import numpy as np

validations = run_validations(
    model,
    tokenizer,
    {eval_layer: saes["next_layer_finetuned_interaction"]},
    validation_dataset.shuffle(seed=42),
    TOKENIZER_BATCH_SIZE,
    EVAL_BATCH_SIZE,
    int(8e3),
    start_layer=eval_layer,
    # end_layer=eval_layer + 1,
    eval_layers=[eval_layer, model.num_layers]
)
print("L0", np.mean(validations.layer_results[eval_layer].l0))
print("KL", np.mean(validations.layer_results[model.num_layers].kl))

tensor([[-0.0000e+00, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
         -0.0000e+00, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
         -0.0000e+00, -0.0000e+0

Running SAE evals:   0%|          | 0/8000 [00:00<?, ?it/s]

L0 191.87221
KL 0.8169567


In [ ]:
model.